# Treatment & Service Demand Visualization

### Milestone 2 – Member 8

*Objective:*  
To visualize the treatment and service demand findings identified during the analysis of laboratory and billing data.

*Data Sources:*
- Lab Results
- Billing

*Visualization Tool:* Plotly

In [101]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

### Libraries Used

- *Pandas:* Used for loading and handling the cleaned datasets.
- *Plotly:* Used to create interactive visualizations.
- *Graph Objects:* Used for KPI cards and the final dashboard layout.

In [102]:
lab = pd.read_csv("../data/processed/lab_results_clean.csv")
billing = pd.read_csv("../data/processed/billing_clean.csv")

print("Lab Results Shape:", lab.shape)
print("Billing Shape:", billing.shape)

Lab Results Shape: (5000, 4)
Billing Shape: (5000, 7)


In [103]:
display(lab.head())
display(billing.head())

,Lab_ID,Admission_ID,Test,Value
0,L00001,A00001,Blood Sugar,91.27
1,L00002,A00002,Blood Sugar,175.69
2,L00003,A00003,Cholesterol,150.60
3,L00004,A00004,Cholesterol,176.53
4,L00005,A00005,Blood Sugar,133.46


,Bill_ID,Admission_ID,Patient_ID,Total,Insurance,Paid,Pending
0,B00001,A00001,P00001,70078,93,70001,77
1,B00002,A00002,P00002,20824,4403,17971,2853
2,B00003,A00003,P00003,6432,239,6228,204
3,B00004,A00004,P00004,82670,9986,77817,4853
4,B00005,A00005,P00005,10376,4009,7571,2805


### Dataset Loading

The cleaned Lab Results and Billing datasets prepared during the data-cleaning stage are used for visualization.

No additional cleaning or modification is performed here. The purpose of this notebook is to visualize the findings from the existing analysis.

In [104]:
print("Lab columns:")
print(lab.columns.tolist())

print("\nBilling columns:")
print(billing.columns.tolist())

Lab columns:
['Lab_ID', 'Admission_ID', 'Test', 'Value']

Billing columns:
['Bill_ID', 'Admission_ID', 'Patient_ID', 'Total', 'Insurance', 'Paid', 'Pending']


## 1. Laboratory Test Demand

### Business Question
*Which laboratory test has the highest demand?*

This visualization represents the test-demand analysis performed in the previous stage.

In [105]:
test_demand = (
    lab["Test"]
    .value_counts()
    .reset_index()
)

test_demand.columns = ["Test", "Number of Tests"]

test_demand

,Test,Number of Tests
0,Blood Sugar,2567
1,Cholesterol,2433


In [106]:
fig_test_demand = px.bar(
    test_demand,
    x="Number of Tests",
    y="Test",
    orientation="h",
    title="Laboratory Test Demand",
    text="Number of Tests"
)

fig_test_demand.update_traces(
    textposition="outside",
    hovertemplate="<b>%{y}</b><br>Number of Tests: %{x:,}<extra></extra>"
)

fig_test_demand.update_layout(
    title_x=0.5,
    xaxis_title="Number of Tests",
    yaxis_title="Laboratory Test",
    height=300,
    margin=dict(l=100, r=80, t=50, b=50)
)

fig_test_demand.show()

### Key Finding

Blood Sugar has the highest demand among the laboratory tests analyzed, with 2,567 tests, compared with 2,433 Cholesterol tests.

## 2. Billing by Laboratory Test

### Business Question
Which laboratory test generates the highest billing?

This visualization represents the billing analysis performed in the previous stage.1

In [107]:
merged_data = pd.merge(
    lab,
    billing,
    on="Admission_ID",
    how="inner"
)

billing_by_test = (
    merged_data.groupby("Test")["Total"]
    .sum()
    .reset_index()
)

billing_by_test.columns = ["Test", "Billing"]

billing_by_test

,Test,Billing
0,Blood Sugar,135814788
1,Cholesterol,127578317


In [108]:
fig_billing = px.treemap(
    billing_by_test,
    path=["Test"],
    values="Billing",
    title="Billing Contribution by Laboratory Test"
)

fig_billing.update_traces(
    texttemplate="%{label}<br>₹%{value:,.0f}",
    hovertemplate="<b>%{label}</b><br>Billing: ₹%{value:,.0f}<extra></extra>"
)

fig_billing.update_layout(
    title_x=0.5,
    height=300,
    margin=dict(l=40, r=20, t=50, b=40)
)

fig_billing.show()

### Key Finding

Blood Sugar generated the highest billing among the laboratory tests analyzed, with total billing of approximately ₹135.81 million, compared with approximately ₹127.58 million for Cholesterol.

## 3. Overall Billing KPIs

The following KPI cards summarize the overall billing position identified during the analysis.

In [109]:
total_billing = billing["Total"].sum()
total_paid = billing["Paid"].sum()
total_pending = billing["Pending"].sum()

paid_percentage = (total_paid / total_billing) * 100
pending_percentage = (total_pending / total_billing) * 100

print(f"Total Billing: ₹{total_billing:,.0f}")
print(f"Total Paid: ₹{total_paid:,.0f}")
print(f"Total Pending: ₹{total_pending:,.0f}")
print(f"Paid Percentage: {paid_percentage:.2f}%")
print(f"Pending Percentage: {pending_percentage:.2f}%")

Total Billing: ₹263,393,105
Total Paid: ₹229,458,756
Total Pending: ₹33,934,349
Paid Percentage: 87.12%
Pending Percentage: 12.88%


## 3.1 Billing KPI Cards

The KPI cards provide a quick overview of the hospital's overall billing, paid amount, and pending amount.

In [110]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig_kpi = make_subplots(
    rows=1,
    cols=3,
    specs=[
        [
            {"type": "indicator"},
            {"type": "indicator"},
            {"type": "indicator"}
        ]
    ],
    horizontal_spacing=0.05
)

# TOTAL BILLING
fig_kpi.add_trace(
    go.Indicator(
        mode="number",
        value=float(total_billing),
        title={
            "text": "<b>TOTAL BILLING</b>",
            "font": {"size": 16}
        },
        number={
            "prefix": "₹",
            "valueformat": ",.0f",
            "font": {"size": 26}
        }
    ),
    row=1,
    col=1
)

# TOTAL PAID
fig_kpi.add_trace(
    go.Indicator(
        mode="number",
        value=float(total_paid),
        title={
            "text": "<b>TOTAL PAID</b>",
            "font": {"size": 16}
        },
        number={
            "prefix": "₹",
            "valueformat": ",.0f",
            "font": {"size": 26}
        }
    ),
    row=1,
    col=2
)

# TOTAL PENDING
fig_kpi.add_trace(
    go.Indicator(
        mode="number",
        value=float(total_pending),
        title={
            "text": "<b>TOTAL PENDING</b>",
            "font": {"size": 16}
        },
        number={
            "prefix": "₹",
            "valueformat": ",.0f",
            "font": {"size": 26}
        }
    ),
    row=1,
    col=3
)

fig_kpi.update_layout(
    height=190,
    margin=dict(
        l=10,
        r=10,
        t=40,
        b=10
    ),
    paper_bgcolor="white",
    plot_bgcolor="white"
)

fig_kpi.show()

In [111]:
print("KPI FIGURE CREATED")
print(fig_kpi)

KPI FIGURE CREATED
Figure({
    'data': [{'domain': {'x': [0.0, 0.3], 'y': [0.0, 1.0]},
              'mode': 'number',
              'number': {'font': {'size': 26}, 'prefix': '₹', 'valueformat': ',.0f'},
              'title': {'font': {'size': 16}, 'text': '<b>TOTAL BILLING</b>'},
              'type': 'indicator',
              'value': 263393105.0},
             {'domain': {'x': [0.35, 0.6499999999999999], 'y': [0.0, 1.0]},
              'mode': 'number',
              'number': {'font': {'size': 26}, 'prefix': '₹', 'valueformat': ',.0f'},
              'title': {'font': {'size': 16}, 'text': '<b>TOTAL PAID</b>'},
              'type': 'indicator',
              'value': 229458756.0},
             {'domain': {'x': [0.7, 1.0], 'y': [0.0, 1.0]},
              'mode': 'number',
              'number': {'font': {'size': 26}, 'prefix': '₹', 'valueformat': ',.0f'},
              'title': {'font': {'size': 16}, 'text': '<b>TOTAL PENDING</b>'},
              'type': 'indicator',
         

In [112]:
html_kpi = plot(
    fig_kpi,
    output_type="div",
    include_plotlyjs=False
)

print("KPI HTML CREATED")
print(len(html_kpi))

KPI HTML CREATED
8115


## 4. Payment Status

### Business Question
What percentage of the total billing has been paid versus pending?

This visualization helps understand the hospital's current payment collection status.

In [113]:
fig_payment = px.pie(
    payment_status,
    names="Status",
    values="Amount",
    hole=0.55,
    title="Paid vs Pending Billing",
    color="Status",
    color_discrete_map={
        "Paid": "green",
        "Pending": "orange"
    }
)

fig_payment.update_traces(
    textinfo="label+percent",
    hovertemplate="<b>%{label}</b><br>Amount: ₹%{value:,.0f}<br>Percentage: %{percent}<extra></extra>"
)

fig_payment.update_layout(
    title_x=0.5,
    height=300,
    margin=dict(l=40, r=20, t=50, b=40)
)

fig_payment.show()

# Final Treatment & Service Demand Dashboard

The following dashboard combines the visualizations created above into a single interactive page.

In [114]:
from plotly.offline import plot

# ============================================
# CONVERT PLOTLY FIGURES TO HTML
# ============================================

html_kpi = plot(
    fig_kpi,
    output_type="div",
    include_plotlyjs="cdn"
)

html_test_demand = plot(
    fig_test_demand,
    output_type="div",
    include_plotlyjs=False
)

html_billing = plot(
    fig_billing,
    output_type="div",
    include_plotlyjs=False
)

html_payment = plot(
    fig_payment,
    output_type="div",
    include_plotlyjs=False
)


# ============================================
# DASHBOARD HTML
# ============================================

dashboard_html = f"""
<!DOCTYPE html>

<html>

<head>

    <meta charset="UTF-8">

    <meta name="viewport"
          content="width=device-width, initial-scale=1.0">

    <title>Treatment & Service Demand Dashboard</title>

    <style>

        * {{
            box-sizing: border-box;
        }}

        body {{
            margin: 0;
            padding: 10px;
            font-family: Arial, sans-serif;
            background: #f5f6f8;
        }}

        .dashboard {{
            width: 100%;
            max-width: 1400px;
            margin: auto;
        }}

        .header {{
            text-align: center;
            margin-bottom: 8px;
        }}

        .header h1 {{
            margin: 3px;
            font-size: 24px;
        }}

        .header p {{
            margin: 3px;
            font-size: 13px;
        }}

        .kpi {{
            background: white;
            border-radius: 8px;
            margin-bottom: 8px;
            width: 100%;
            height: 190px;
            padding: 0;
            overflow: hidden;
        }}

        .kpi .plotly-graph-div {{
            width: 100% !important;
            height: 190px !important;
        }}

        .charts {{
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 8px;
            width: 100%;
        }}

        .chart {{
            background: white;
            border-radius: 8px;
            width: 100%;
            height: 300px;
            padding: 2px;
            overflow: visible;
        }}

        .chart .plotly-graph-div {{
            width: 100% !important;
            height: 100% !important;
        }}

        @media (max-width: 900px) {{

            .header h1 {{
                font-size: 20px;
            }}

            .charts {{
                grid-template-columns: 1fr;
            }}

            .chart {{
                height: 300px;
            }}

            .kpi {{
                height: 190px;
            }}

            .kpi .plotly-graph-div {{
                height: 190px !important;
            }}

        }}

    </style>

</head>

<body>

<div class="dashboard">

    <!-- HEADER -->

    <div class="header">

        <h1>
            Treatment & Service Demand Dashboard
        </h1>

        <p>
            Laboratory Test Demand and Billing Analysis
        </p>

    </div>


    <!-- KPI -->

    <div class="kpi">

        {html_kpi}

    </div>


    <!-- CHARTS -->

    <div class="charts">

        <div class="chart">
            {html_test_demand}
        </div>

        <div class="chart">
            {html_billing}
        </div>

        <div class="chart">
            {html_payment}
        </div>

    </div>

</div>

</body>

</html>
"""


# ============================================
# SAVE DASHBOARD
# ============================================

with open(
    "../reports/treatment_service_demand_dashboard.html",
    "w",
    encoding="utf-8"
) as f:

    f.write(dashboard_html)

print("Dashboard created successfully!")

Dashboard created successfully!
